In [32]:
from cresowlve.utils import read_json

en_bench = read_json("../experiments/data/task/chgk_en_benchmark.json")
ru_bench = read_json("../experiments/data/task/chgk_ru_benchmark.json")
reasoning_types_results = read_json("../experiments/data/task/chgk_benchmark_reasoning_types.json")
domain_results = read_json("../experiments/data/task/chgk_benchmark_domain.json")
culture_results = read_json("../experiments/data/task/chgk_benchmark_culture_lang.json")

final_en_bench = []
final_ru_bench = []

for sample in en_bench["data"]:
    reasoning_type_sample = next(item for item in reasoning_types_results["data"] if item["id"] == sample["id"])
    culture_sample = next(item for item in culture_results["data"] if item["id"] == sample["id"])
    domain_sample = next(item for item in domain_results["data"] if item["id"] == sample["id"])

    if reasoning_type_sample["reasoning_type"] == "creative":
        final_en_bench.append({
            "id": sample["id"],
            "question": sample["question"].replace("\n", " ").strip(),
            "answer": sample["answer"].replace("\n", " ").strip(),
            "difficulty": sample["difficulty_id"],
            "explanation": sample["comment"].replace("\n", " ").strip() if sample["comment"] and sample["comment"].lower() != "none" else None,
            "other_answers": sample["notes"].replace("\n", " ").strip() if sample["notes"] and sample["notes"].lower() != "none" else None,
            "knowledge_domains": domain_sample.get("grouped_domains", []),
            "creative_domains": reasoning_type_sample.get("concepts", []),
            "cultures": culture_sample.get("culture_langs", [])
        })

for sample in ru_bench["data"]:
    reasoning_type_sample = next(item for item in reasoning_types_results["data"] if item["id"] == sample["id"])
    culture_sample = next(item for item in culture_results["data"] if item["id"] == sample["id"])
    domain_sample = next(item for item in domain_results["data"] if item["id"] == sample["id"])

    if reasoning_type_sample["reasoning_type"] == "creative":
        final_ru_bench.append({
            "id": sample["id"],
            "question": sample["question"].replace("\n", " ").strip(),
            "answer": sample["answer"].replace("\n", " ").strip(),
            "difficulty": sample["difficulty_id"],
            "explanation": sample["comment"].replace("\n", " ").strip() if sample["comment"] and sample["comment"].lower() != "none" else None,
            "other_answers": sample["notes"].replace("\n", " ").strip() if sample["notes"] and sample["notes"].lower() != "none" else None,
            "knowledge_domains": domain_sample.get("grouped_domains", []),
            "creative_domains": reasoning_type_sample.get("concepts", []),
            "cultures": culture_sample.get("culture_langs", [])
        })

In [33]:
len(final_en_bench), len(final_ru_bench)

(2061, 2061)

In [34]:
final_en_bench[:3]

[{'id': 'chgk_83a712ed75',
  'question': 'In a wartime novel, the hero looks at the sky, where contrails of airplanes and bursts of anti-aircraft shells form an unpleasant, in his opinion, picture. What word did we replace with "picture"?',
  'answer': 'Melody.',
  'difficulty': 3,
  'explanation': 'Contrails are the lines of a musical staff, and the bursts are the symbols of notes.',
  'other_answers': 'Music, score, cacophony.',
  'knowledge_domains': ['Visual Arts', 'Literature', 'Music', 'Military'],
  'creative_domains': ['metaphor', 'lateral thinking'],
  'cultures': ['english']},
 {'id': 'chgk_e66ed999fd',
  'question': 'Addressing one of his friends before his death, he used a word that means not only "farewell" but also "with God." In his native language, this word and the friend\'s nickname start with the same letter. Name both the word and the nickname.',
  'answer': 'Adieu, Aramis.',
  'difficulty': 3,
  'explanation': 'D\'Artagnan said before his death: "Athos, Porthos, au

In [35]:
import os
from datasets import Dataset
from dotenv import load_dotenv

load_dotenv()

# Set these in your environment or replace the default value below.
repo_id = "mismayil/cresowlve"

en_dataset = Dataset.from_list(final_en_bench)
ru_dataset = Dataset.from_list(final_ru_bench)

# Push two subsets (dataset configs) to one dataset repo: en and ru.
en_dataset.push_to_hub(repo_id=repo_id, config_name="en", split="test", private=True)
ru_dataset.push_to_hub(repo_id=repo_id, config_name="ru", split="test", private=True)

print(f"Published to https://huggingface.co/datasets/{repo_id}")
print("Available subsets/configs: en, ru")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Published to https://huggingface.co/datasets/mismayil/cresowlve
Available subsets/configs: en, ru


In [20]:
from collections import Counter
domain_results = read_json("../experiments/data/task/chgk_benchmark_domain.json")
domains = Counter([domain for sample in domain_results["data"] for domain in sample["grouped_domains"]])
# sort by count
domains = dict(sorted(domains.items(), key=lambda item: item[1], reverse=True))

In [21]:
", ".join(domains)

'Literature, History, Film & Media Studies, Languages & Linguistics, Human Geography, Religious Studies, Anthropology, Physical Education & Sports, Biology, Engineering & Technology, Visual Arts, Music, Political Science, Home Economics & Daily Life, Performing Arts, Psychology, Sociology, Earth & Environmental Science, Military, Physics, Astronomy & Space Science, Business Studies, Philosophy, Design & Architecture, Medicine & Health Sciences, Economics, Mathematics, Chemistry, Law & Criminology, Science, Art History & Visual Culture, Education, Communication, Archaeology'

In [26]:
creative_domain_results = read_json("../experiments/data/task/chgk_benchmark_reasoning_types.json")
creative_domains = Counter([concept for sample in creative_domain_results["data"] for concept in sample.get("concepts", [])])

# sort by count
creative_domains = sorted(creative_domains, key=lambda x: creative_domains.get(x, 0), reverse=True)

In [27]:
", ".join(creative_domains)

'lateral thinking, analogy, abstraction, joke, pun, metaphor, commonsense reasoning, poem, idiom, neologism, sarcasm, proverb, divergent thinking, compositionality, simile, historical reference, hyperbole, cultural reference, literary reference, reinterpretation, myth, irony, allegory, ambiguity'

In [28]:
culture_results = read_json("../experiments/data/task/chgk_benchmark_culture_lang.json")
culture_domains = Counter([culture for sample in culture_results["data"] for culture in sample.get("culture_langs", [])])

# sort by count
culture_domains = sorted(culture_domains, key=lambda x: culture_domains.get(x, 0), reverse=True)

In [31]:
", ".join([culture.capitalize() for culture in culture_domains])

'English, Russian, French, German, Italian, Greek, Latin, American, Spanish, Japanese, Polish, Arabic, Dutch, Swedish, Chinese, Hebrew, Ukrainian, Roman, Indian, Norwegian, Danish, Scottish, Portuguese, Turkish, Czech, Swiss, Egyptian, Georgian, Irish, Persian, Brazilian, European, Armenian, Biblical, Hungarian, Finnish, Christian, Scandinavian, Lithuanian, Romanian, Bulgarian, Christianity, Serbian, None, Canadian, Korean, Vietnamese, Sanskrit, Australian, Hindi, Azerbaijani, Norse, Hindu, Icelandic, Slavic, Islamic, Latvian, Ethiopian, Catalan, African, Nepali, Swahili, Tatar, Peruvian, Belgian, Belarusian, Native american, Afrikaans, Byzantine, Aztec, South african, Medieval european, Iranian, Jamaican, Croatian, Argentinian, Tibetan, Eskimo, Venezuelan, Slovenian, Catholic, West african, Filipino, Maltese, Chilean, Polynesian, Kyrgyz, Inuit, Esperanto, Cuban, Mongolian, Soviet, Aramaic, Indonesian, Dyirbal, Pakistani, Malay, Nigerian, Vanuatu, Berber, Phoenician, Indigenous siberia